# HPA Data Access with `astroquery` 

```
Author: ESDC Team at ESAC
Date Last Modified: 20/08/2026
```

This notebook demonstrate programmatic access to the data in ESA's [HPA](https://hpa.esa.int) using the Table Access Protocol (TAP) and the Astronomical Query Language (ADQL) with the [astroquery](https://astroquery.readthedocs.io/en/latest/index.html) `python` package.

Accessing the HPA via `astroquery` has the following *advantages* (+) and *disadvantages* (-) compared to `PyVO` and `sunpy`:

+ `astroquery` allows to query a large variety of astronomical services, beyond those that are accessible via TAP
+ `astroquery` adds a simplification layer on top for most services, enabling syntax with simpler, "pythonic" queries rather than ADQL

- The simplification layer is not available for all missions, requiring to revert back to TAP queries
- Synchronous queries return a maximum of 2000 rows. If your request might result in more rows, you need to run an asynchronous query.a `astroquery` does not tell you if you only retrieved a partial table via a synchrnous query.

Refer to the other tutorials in the [Getting Started](https://github.com/HPA-ESDC-ESA-INT/HPA-Jupyter-Notebooks/tree/main/getting_started) section for more information on `PyVO` and `sunpy`.

## Table of Contents

- [1. Basic Use](#1-basic-use)
  - [1.1. Getting table metadata](#11-getting-table-metadata)
  - [1.2. Getting table data: Synchronous query](#12-getting-table-data-synchronous-query)
  - [1.3. Asynchronous query](#13-asynchronous-query)
  - [1.4. Asynchronous job removal](#14-asynchronous-job-removal)



## Requirements

The following python packages are required: `astroquery`, `numpy`, `ipython`. You can install them by running the line below in your terminal (after removing the leading `#`).

In [ ]:
# python -m pip install astroquery numpy IPython 

In [73]:
from astroquery.utils.tap.core import TapPlus
from IPython.display import Image, Video
import numpy as np

## 1. Basic Use

We access the P2SA archive using the TAP+ protocol. TAP+ offers all funcationality that TAP does, but also allows users to log in to remote servers and have persistent user sessions. We do not make use of this functionality here, and instead use the anynomous user for all our requests. This means that our query results are stored temporarily on the ESDC servers and are then remove automatically.

To connect via TAP, we need the service's TAP URL. We find this on the [P2SA documentation](https://p2sa.esac.esa.int/p2sa/#aio).

### 1.1 Getting table metadata

First, we want to see which tables are accessible to use. We load the names of all public tables. Note that we are making **metadata** queries here, we are not yet retrieving the data itself.

In [74]:
p2sa = TapPlus(url='https://p2sa.esac.esa.int/p2sa-sl-tap/tap')  # we got this URL from the P2SA documentation
tables = p2sa.load_tables()

for table in tables:
    print(table.get_qualified_name())

INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]
p2sa.file
p2sa.full_disk_solar_image
p2sa.instrument
p2sa.lyra_observation
p2sa.observation
p2sa.observatory
p2sa.science_object
p2sa.swap_observation
p2sa.v_carrington_rotation_file
p2sa.v_file
p2sa.v_lyra_observation
p2sa.v_observation
p2sa.v_swap_observation
public.dual
tap_config.coord_sys
tap_config.properties
tap_schema.columns
tap_schema.key_columns
tap_schema.keys
tap_schema.schemas
tap_schema.tables


The relevant tables for us have a `p2sa` prefix. Next, we load a table and inspect the column names.

In [75]:
table = p2sa.load_table('p2sa.observation')
print('TAP Table name:', table.get_qualified_name())

columns_observation = [column.name for column in table.columns]
print(columns_observation)


TAP Table name: p2sa.observation
['begin_date', 'calibrated', 'end_date', 'file_format', 'file_name', 'file_path', 'file_size', 'instrument_oid', 'observation_oid', 'observation_type', 'processing_level', 'science_objective', 'science_object_oid', 'wavelength_range']


This table contains metadata of Proba-2 observations, including start and end dates, calibration levels, filenames, and instrument IDs. In the P2Sa, there is also a `v_observation` table. To see how it differs, we retrieve the table and compare the columns between the two tables.

In [77]:
table = p2sa.load_table('p2sa.v_observation')
print('TAP Table name:', table.get_qualified_name())

columns_v_observation = [column.name for column in table.columns]
print(columns_v_observation)

difference = set(columns_v_observation) - set(columns_observation)
print("Columns in p2sa.v_observation but not in p2sa.observation:", difference)

TAP Table name: p2sa.v_observation
['begin_date', 'calibrated', 'end_date', 'file_format', 'file_name', 'file_path', 'file_size', 'instrument_name', 'observation_oid', 'observation_type', 'observatory_name', 'processing_level', 'science_objective', 'science_object_name', 'science_object_oid', 'wavelength_range']
Columns in p2sa.v_observation but not in p2sa.observation: {'observatory_name', 'science_object_name', 'instrument_name'}


`v_observation` is slightly more human-readable than `observation`, as it contains the instrument, observatory, and science object names as well as their numeric IDs.

### 1.2. Getting table data: Synchronous query

To get the actual content of these tables, we use the `launch_job`. We can run ADQL queries both synchronously and asynchronously. Synchronous queries are good for small queries that execute fast. The output is limited to 2000 rows.

We query the first 100 rows of the `p2sa.observation` table. We request all columns (`*`).

In [ ]:
job = p2sa.launch_job("select top 100 * from p2sa.observation")
print(job)


<Table length=100>
       name        dtype 
------------------ ------
        begin_date object
        calibrated   bool
          end_date object
       file_format  str20
         file_name object
         file_path  str80
         file_size  int64
    instrument_oid  int32
   observation_oid  int32
  observation_type object
  processing_level str200
 science_objective object
science_object_oid  int32
  wavelength_range object
Jobid: None
Phase: COMPLETED
Owner: None
Output file: 1784107633032OPER-result.vot
Results: None


`launch_job` returns a `job` that, when printed, shows us that we received the expected 100 entries in the table and the expected columns. The job has completed and the results are stored in memory - meaning that they are lost once our python session ends. To store them to file, we use the following call.

In [79]:
job = p2sa.launch_job("select top 100 * from p2sa.observation", dump_to_file=True, output_file="p2sa_observation_results.vot.gz")
print(job)

Jobid: None
Phase: COMPLETED
Owner: None
Output file: p2sa_observation_results.vot.gz
Results: None


To inspect the contents of the returned table, we use `get_results`.

In [80]:
result = job.get_results()
print(result)

print(result['file_name'].tolist())


       begin_date       calibrated ... science_object_oid wavelength_range
----------------------- ---------- ... ------------------ ----------------
2010-08-24T04:37:44.714      False ...                  1              174
2010-08-24T05:53:44.779      False ...                  1              174
2010-12-19T13:28:42.835      False ...                  1              174
2010-12-19T11:25:21.736      False ...                  1              174
2010-12-19T09:25:15.588      False ...                  1              174
2010-12-19T06:47:32.513      False ...                  1              174
2010-12-19T05:21:01.389      False ...                  1              174
2010-12-19T03:38:15.361      False ...                  1              174
2010-12-19T02:08:29.235      False ...                  1              174
2010-12-19T01:13:14.191      False ...                  1              174
                    ...        ... ...                ...              ...
2010-08-24T10:09:00.005  

### 1.3. Asynchronous query

Asynchronous TAP queries run as background jobs on the server. After submission, a job identifier is returned that can be used to check the query status and download the results when ready. This mode is better suited than synchronous execution for large or complex queries because it is not limited by HTTP timeouts and can handle longer processing times.

We request the first 5000 rows of the `p2sa.observation` table. Note that this is not possible with synchronous queries, as they are limited to 2000 rows.

In [81]:
job = p2sa.launch_job_async("select top 5000 * from p2sa.observation")
print(job)


INFO: Query finished. [astroquery.utils.tap.core]
<Table length=5000>
       name        dtype 
------------------ ------
        begin_date object
        calibrated   bool
          end_date object
       file_format  str20
         file_name object
         file_path  str80
         file_size  int64
    instrument_oid  int32
   observation_oid  int32
  observation_type object
  processing_level str200
 science_objective object
science_object_oid  int32
  wavelength_range object
Jobid: 1784107946873OPER
Phase: COMPLETED
Owner: None
Output file: async_20260715093226.vot
Results: None


This query is quite easy and finishes fast. More complicated queries (see below) will take longer to execute. In this case, `job.get_results` will wait for the server-side execution to finish before returning the results.

### 1.4. Asynchronous job removal

Each job has an associated `job.jobid`. It can be used to remove asynchronous jobs from the server.

In [82]:
p2sa.remove_jobs([job.jobid])

INFO: Removed jobs: '['1784107946873OPER']'. [astroquery.utils.tap.core]
